In [ ]:
!pip -q install ultralytics fastapi uvicorn nest-asyncio pyngrok

In [ ]:
import cv2
import os
import math
import time
import sqlite3
import threading
import nest_asyncio
import numpy as np
import pandas as pd

from datetime import datetime
from collections import defaultdict

from ultralytics import YOLO

print("Libraries loaded successfully.")

In [ ]:
from google.colab import files

uploaded = files.upload()

VIDEO_PATH = list(uploaded.keys())[0]

print("Video:", VIDEO_PATH)

In [ ]:
model = YOLO("yolo11n.pt")

print("YOLO model loaded.")

Inspect CCTV video

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise Exception("Could not open video.")

WIDTH = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS = cap.get(cv2.CAP_PROP_FPS)
FRAME_COUNT = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

DURATION = FRAME_COUNT / FPS if FPS > 0 else 0

print("Width:", WIDTH)
print("Height:", HEIGHT)
print("FPS:", FPS)
print("Frames:", FRAME_COUNT)
print("Duration:", round(DURATION, 2), "seconds")

cap.release()

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Entry/Exit virtual line
LINE_Y = int(HEIGHT * 0.55)

# Store zones
ZONE_A = [
    (0, 0),
    (WIDTH // 2, 0),
    (WIDTH // 2, HEIGHT // 2),
    (0, HEIGHT // 2)
]

ZONE_B = [
    (WIDTH // 2, 0),
    (WIDTH, 0),
    (WIDTH, HEIGHT // 2),
    (WIDTH // 2, HEIGHT // 2)
]

ZONE_C = [
    (0, HEIGHT // 2),
    (WIDTH // 2, HEIGHT // 2),
    (WIDTH // 2, HEIGHT),
    (0, HEIGHT)
]

ZONE_D = [
    (WIDTH // 2, HEIGHT // 2),
    (WIDTH, HEIGHT // 2),
    (WIDTH, HEIGHT),
    (WIDTH // 2, HEIGHT)
]

ZONES = {
    "Electronics": ZONE_A,
    "Clothing": ZONE_B,
    "Grocery": ZONE_C,
    "Cosmetics": ZONE_D
}

print("Configuration ready.")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Entry/Exit virtual line
LINE_P1 = (0, LINE_Y)
LINE_P2 = (WIDTH, LINE_Y)

# Store zones
ZONE_A = [
    (0, 0),
    (WIDTH // 2, 0),
    (WIDTH // 2, HEIGHT // 2),
    (0, HEIGHT // 2)
]

ZONE_B = [
    (WIDTH // 2, 0),
    (WIDTH, 0),
    (WIDTH, HEIGHT // 2),
    (WIDTH // 2, HEIGHT // 2)
]

ZONE_C = [
    (0, HEIGHT // 2),
    (WIDTH // 2, HEIGHT // 2),
    (WIDTH // 2, HEIGHT),
    (0, HEIGHT)
]

ZONE_D = [
    (WIDTH // 2, HEIGHT // 2),
    (WIDTH, HEIGHT // 2),
    (WIDTH, HEIGHT),
    (WIDTH // 2, HEIGHT)
]

ZONES = {
    "Entrance": ZONE_A,
    "Product Area": ZONE_B,
    "Billing": ZONE_C,
    "Exit": ZONE_D
}

print("Configuration ready.")

In [ ]:
DB_PATH = "retailvision.db"

conn = sqlite3.connect(DB_PATH, check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS events (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    event_type TEXT,
    track_id INTEGER,
    zone TEXT,
    confidence REAL,
    timestamp TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS zone_visits (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    track_id INTEGER,
    zone TEXT,
    entry_time TEXT,
    exit_time TEXT,
    duration_seconds REAL
)
""")

conn.commit()

print("Database initialized.")

In [ ]:
def point_in_polygon(point, polygon):
    polygon_np = np.array(polygon, np.int32)

    return cv2.pointPolygonTest(
        polygon_np,
        point,
        False
    ) >= 0


def get_zone(point):
    for zone_name, polygon in ZONES.items():
        if point_in_polygon(point, polygon):
            return zone_name

    return "Outside"


def draw_zone(frame, polygon, name):
    pts = np.array(polygon, np.int32)

    cv2.polylines(
        frame,
        [pts],
        True,
        (255, 255, 255),
        2
    )

    x, y = polygon[0]

    cv2.putText(
        frame,
        name,
        (x + 10, y + 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

In [ ]:
track_history = {}

entry_count = 0
exit_count = 0

zone_counts = defaultdict(int)

zone_start_times = {}

dwell_times = defaultdict(list)

events = []

previous_positions = {}

frame_number = 0

print("Analytics engine initialized.")

In [ ]:
OUTPUT_VIDEO = "retailvision_output.mp4"

cap = cv2.VideoCapture(VIDEO_PATH)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    OUTPUT_VIDEO,
    fourcc,
    FPS,
    (WIDTH, HEIGHT)
)

print("Starting CCTV processing...")

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1

    # --------------------------------------------------------
    # YOLO TRACKING
    # --------------------------------------------------------

    results = model.track(
        frame,
        persist=True,
        classes=[0],        # person only
        tracker="bytetrack.yaml",
        verbose=False
    )

    current_zone_counts = defaultdict(int)

    current_time = frame_number / FPS

    # --------------------------------------------------------
    # PROCESS DETECTIONS
    # --------------------------------------------------------

    if results[0].boxes is not None:

        boxes = results[0].boxes

        if boxes.id is not None:

            ids = boxes.id.cpu().numpy().astype(int)

            xyxy = boxes.xyxy.cpu().numpy()

            confidences = boxes.conf.cpu().numpy()

            for box, track_id, confidence in zip(
                xyxy,
                ids,
                confidences
            ):

                x1, y1, x2, y2 = map(int, box)

                # Bottom-center of person
                center_x = int((x1 + x2) / 2)
                center_y = int(y2)

                current_point = (center_x, center_y)

                # ------------------------------------------------
                # DRAW PERSON
                # ------------------------------------------------

                cv2.rectangle(
                    frame,
                    (x1, y1),
                    (x2, y2),
                    (255, 255, 255),
                    2
                )

                cv2.circle(
                    frame,
                    current_point,
                    5,
                    (255, 255, 255),
                    -1
                )

                cv2.putText(
                    frame,
                    f"ID {track_id}",
                    (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (255, 255, 255),
                    2
                )

                # ------------------------------------------------
                # ZONE DETECTION
                # ------------------------------------------------

                zone = get_zone(current_point)

                if zone != "Outside":

                    current_zone_counts[zone] += 1

                    if track_id not in zone_start_times:

                        zone_start_times[track_id] = {
                            zone: current_time
                        }

                    elif zone not in zone_start_times[track_id]:

                        zone_start_times[track_id][zone] = current_time

                    # Calculate dwell time

                    start = zone_start_times[track_id][zone]

                    dwell = current_time - start

                    dwell_times[zone].append(dwell)

                # ------------------------------------------------
                # ENTRY / EXIT DETECTION
                # ------------------------------------------------

                if track_id in previous_positions:

                    previous_y = previous_positions[track_id]

                    # Entering
                    if previous_y < LINE_Y <= center_y:

                        entry_count += 1

                        timestamp = datetime.now().isoformat()

                        events.append({
                            "event_type": "ENTRY",
                            "track_id": int(track_id),
                            "zone": zone,
                            "confidence": float(confidence),
                            "timestamp": timestamp
                        })

                        cursor.execute("""
                        INSERT INTO events
                        (event_type, track_id, zone, confidence, timestamp)
                        VALUES (?, ?, ?, ?, ?)
                        """, (
                            "ENTRY",
                            int(track_id),
                            zone,
                            float(confidence),
                            timestamp
                        ))

                    # Exiting
                    elif previous_y > LINE_Y >= center_y:

                        exit_count += 1

                        timestamp = datetime.now().isoformat()

                        events.append({
                            "event_type": "EXIT",
                            "track_id": int(track_id),
                            "zone": zone,
                            "confidence": float(confidence),
                            "timestamp": timestamp
                        })

                        cursor.execute("""
                        INSERT INTO events
                        (event_type, track_id, zone, confidence, timestamp)
                        VALUES (?, ?, ?, ?, ?)
                        """, (
                            "EXIT",
                            int(track_id),
                            zone,
                            float(confidence),
                            timestamp
                        ))

                previous_positions[track_id] = center_y

    # --------------------------------------------------------
    # DRAW STORE ZONES
    # --------------------------------------------------------

    for zone_name, polygon in ZONES.items():

        draw_zone(
            frame,
            polygon,
            zone_name
        )

    # --------------------------------------------------------
    # DRAW ENTRY / EXIT LINE
    # --------------------------------------------------------

    cv2.line(
        frame,
        (0, LINE_Y),
        (WIDTH, LINE_Y),
        (255, 255, 255),
        3
    )

    cv2.putText(
        frame,
        "ENTRY / EXIT LINE",
        (20, LINE_Y - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    # --------------------------------------------------------
    # DASHBOARD
    # --------------------------------------------------------

    current_customers = max(
        0,
        entry_count - exit_count
    )

    overlay = frame.copy()

    cv2.rectangle(
        overlay,
        (10, 10),
        (390, 180),
        (0, 0, 0),
        -1
    )

    frame = cv2.addWeighted(
        overlay,
        0.65,
        frame,
        0.35,
        0
    )

    cv2.putText(
        frame,
        "RETAILVISION AI",
        (25, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        f"Entries: {entry_count}",
        (25, 75),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        f"Exits: {exit_count}",
        (25, 105),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        f"Current Customers: {current_customers}",
        (25, 135),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        f"Frame: {frame_number}",
        (25, 165),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 255, 255),
        2
    )

    # --------------------------------------------------------
    # WRITE OUTPUT
    # --------------------------------------------------------

    out.write(frame)

    # Commit periodically

    if frame_number % 100 == 0:
        conn.commit()

cap.release()
out.release()

conn.commit()

print("Processing completed.")
print("Output:", OUTPUT_VIDEO)

In [ ]:
from IPython.display import HTML
from base64 import b64encode

def display_video(path):

    with open(path, "rb") as f:
        video_data = f.read()

    data_url = "data:video/mp4;base64," + b64encode(video_data).decode()

    return HTML(
        f"""
        <video width="800" controls>
            <source src="{data_url}" type="video/mp4">
        </video>
        """
    )

display_video(OUTPUT_VIDEO)

In [ ]:
events_df = pd.DataFrame(events)

if len(events_df) > 0:

    print("Event Summary")
    display(
        events_df.head(20)
    )

else:

    print("No entry/exit events detected.")

Zone analytics

In [ ]:
zone_summary = []

for zone, values in dwell_times.items():

    if len(values) > 0:

        zone_summary.append({
            "Zone": zone,
            "Visits": len(values),
            "Average Dwell Time (sec)": round(
                np.mean(values),
                2
            ),
            "Maximum Dwell Time (sec)": round(
                np.max(values),
                2
            )
        })

zone_df = pd.DataFrame(zone_summary)

if len(zone_df) > 0:
    display(zone_df)

else:
    print("No zone data available.")

In [ ]:
if len(events_df) > 0:

    events_df.to_csv(
        "retailvision_events.csv",
        index=False
    )

if len(zone_df) > 0:

    zone_df.to_csv(
        "retailvision_zone_analytics.csv",
        index=False
    )

print("Reports generated.")

In [ ]:
total_entries = entry_count
total_exits = exit_count

current_customers = max(
    0,
    total_entries - total_exits
)

peak_occupancy = current_customers

if len(events_df) > 0:

    entry_events = events_df[
        events_df["event_type"] == "ENTRY"
    ]

    if len(entry_events) > 0:

        hourly_entries = pd.to_datetime(
            entry_events["timestamp"]
        ).dt.hour.value_counts()

        peak_hour = hourly_entries.idxmax()

    else:

        peak_hour = "N/A"

else:

    peak_hour = "N/A"


print("=" * 50)
print("        RETAILVISION AI REPORT")
print("=" * 50)

print("Total Entries       :", total_entries)
print("Total Exits         :", total_exits)
print("Current Customers   :", current_customers)
print("Peak Hour           :", peak_hour)

if len(zone_df) > 0:

    most_visited = zone_df.loc[
        zone_df["Visits"].idxmax(),
        "Zone"
    ]

    print("Most Visited Zone   :", most_visited)

print("=" * 50)

In [ ]:
from fastapi import FastAPI
from fastapi.responses import JSONResponse
import uvicorn

app = FastAPI(
    title="RetailVision AI API",
    description="AI-powered retail CCTV analytics API",
    version="1.0"
)


@app.get("/")
def home():

    return {
        "project": "RetailVision AI",
        "status": "running",
        "description": "Retail CCTV Customer Analytics System"
    }


@app.get("/api/status")
def status():

    return {
        "total_entries": entry_count,
        "total_exits": exit_count,
        "current_customers": max(
            0,
            entry_count - exit_count
        )
    }


@app.get("/api/events")
def get_events():

    conn_api = sqlite3.connect(DB_PATH)

    df = pd.read_sql_query(
        "SELECT * FROM events ORDER BY id DESC",
        conn_api
    )

    conn_api.close()

    return df.to_dict(
        orient="records"
    )


@app.get("/api/zones")
def get_zones():

    conn_api = sqlite3.connect(DB_PATH)

    df = pd.read_sql_query(
        "SELECT * FROM zone_visits ORDER BY id DESC",
        conn_api
    )

    conn_api.close()

    return df.to_dict(
        orient="records"
    )


@app.get("/api/analytics")
def analytics():

    return {
        "entries": entry_count,
        "exits": exit_count,
        "current_customers": max(
            0,
            entry_count - exit_count
        ),
        "zones": {
            zone: len(values)
            for zone, values in dwell_times.items()
        }
    }


print("FastAPI application created.")

In [ ]:
nest_asyncio.apply()

def run_api():

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )

api_thread = threading.Thread(
    target=run_api,
    daemon=True
)

api_thread.start()

time.sleep(3)

print("FastAPI server running on port 8000")

In [ ]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/api/status"
)

print(response.json())

In [ ]:
from google.colab.output import eval_js

url = eval_js(
    "google.colab.kernel.proxyPort(8000)"
)

print(url + "/docs")

In [ ]:
from google.colab import files

files.download(
    OUTPUT_VIDEO
)